# 06 — Innovation stress tests

Apply identical volatility and downside-tail perturbations to every fitted forecast. Robustness is measured as degradation from the unstressed case.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.api.stress import stress_test

In [2]:
CACHE = ROOT / 'results/notebook_cache'
split = np.load(CACHE / 'financial_split.npz')
var_data = np.load(CACHE / 'fitted_var.npz')
fitted_var = {'beta': var_data['beta'], 'lags': int(var_data['lags']), 'include_intercept': True}
history = np.vstack([split['train'], split['calibration']])[-fitted_var['lags']:]
forecasts = {}
for name in ('gaussian', 'student_t', 'bootstrap', 'diffusion'):
    path = CACHE / f'forecast_{name}.npz'
    if path.exists():
        data = np.load(path)
        forecasts[name] = {key: data[key] for key in data.files}
assert forecasts, 'Run notebooks 03 and 04 first.'

In [3]:
scale_results = stress_test(
    fitted_var, forecasts, history, split['test'], method='scale',
    epsilon_grid=[0.0, 0.1, 0.2, 0.4], dgp_name='financial',
)
tail_results = stress_test(
    fitted_var, forecasts, history, split['test'], method='tail',
    epsilon_grid=[
        {'prob': 0.0, 'multiplier': 5.0},
        {'prob': 0.02, 'multiplier': 5.0},
        {'prob': 0.05, 'multiplier': 5.0},
        {'prob': 0.10, 'multiplier': 5.0},
    ], dgp_name='financial',
)

In [4]:
display(scale_results[['innovation_model', 'epsilon_value', 'energy_score_degradation', 'ece_degradation']])
display(tail_results[['innovation_model', 'contamination_prob', 'energy_score_degradation', 'ece_degradation']])
scale_results.to_csv(CACHE / 'scale_stress.csv', index=False)
tail_results.to_csv(CACHE / 'tail_stress.csv', index=False)

,innovation_model,epsilon_value,energy_score_degradation,ece_degradation
0,bootstrap,0.0,0.000000,0.000000
1,bootstrap,0.1,0.000111,0.025069
2,bootstrap,0.2,0.000283,0.056061
3,bootstrap,0.4,0.000773,0.102204
4,diffusion,0.0,0.000000,0.000000
5,diffusion,0.1,0.000283,0.031680
6,diffusion,0.2,0.000627,0.057851
7,diffusion,0.4,0.001459,0.088843
8,gaussian,0.0,0.000000,0.000000
9,gaussian,0.1,0.000111,0.028788


,innovation_model,contamination_prob,energy_score_degradation,ece_degradation
0,bootstrap,0.00,0.000000,0.000000
1,bootstrap,0.02,0.000066,0.005096
2,bootstrap,0.05,0.000235,0.020248
3,bootstrap,0.10,0.000681,0.042975
4,diffusion,0.00,0.000000,0.000000
5,diffusion,0.02,0.000166,0.014463
6,diffusion,0.05,0.000489,0.025482
7,diffusion,0.10,0.001195,0.049587
8,gaussian,0.00,0.000000,0.000000
9,gaussian,0.02,0.000081,0.006749
